In [ ]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
import math

In [54]:
class QuadraticState(TypedDict):
    a: float
    b: float
    c: float
    equation: str
    discriminant: float
    result: str

In [55]:
def show_equation(state: QuadraticState) -> dict:
    a, b, c = state["a"], state["b"], state["c"]

    if a == 1:
        a_part = "x²"
    elif a == -1:
        a_part = "-x²"
    elif a == 0:
        a_part = ""
    else:
        a_part = f"{int(a)}x²"

    if b == 0:
        b_part = ""
    elif b == 1:
        b_part = " + x"
    elif b == -1:
        b_part = " - x"
    elif b > 0:
        b_part = f" + {int(b)}x"
    else:
        b_part = f" - {int(abs(b))}x"

    if c == 0:
        c_part = ""
    elif c > 0:
        c_part = f" + {int(c)}"
    else:
        c_part = f" - {int(abs(c))}"

    equation = f"{a_part}{b_part}{c_part}".strip()

    if equation == "":
        equation = "0"

    return {"equation": equation}

In [56]:
def calculate_discriminant(state: QuadraticState) -> dict:
    a, b, c = state["a"], state["b"], state["c"]
    d = (b ** 2) - (4 * a * c)
    return {"discriminant": d}

In [57]:
def check_condition(state: QuadraticState) -> Literal["no_real_roots", "real_roots", "repeated_roots"]:
    d = state["discriminant"]
    if d < 0:
        return "no_real_roots"
    elif d > 0:
        return "real_roots"
    else:
        return "repeated_roots"

In [58]:
def no_real_roots(state: QuadraticState) -> dict:
    return {"result": "No real roots exist (discriminant is negative)."}


def real_roots(state: QuadraticState) -> dict:
    a, b, c = state["a"], state["b"], state["c"]
    d = state["discriminant"]
    root1 = (-b + math.sqrt(d)) / (2 * a)
    root2 = (-b - math.sqrt(d)) / (2 * a)
    return {"result": f"Two real roots: {root1:.2f} and {root2:.2f}"}


def repeated_roots(state: QuadraticState) -> dict:
    a, b = state["a"], state["b"]
    root = -b / (2 * a)
    return {"result": f"One repeated root: {root:.2f}"}

In [59]:
builder = StateGraph(QuadraticState)

# Nodes
builder.add_node("show_equation", show_equation)
builder.add_node("calculate_discriminant", calculate_discriminant)
builder.add_node("no_real_roots", no_real_roots)
builder.add_node("real_roots", real_roots)
builder.add_node("repeated_roots", repeated_roots)

# Edges
builder.add_edge(START, "show_equation")
builder.add_edge("show_equation", "calculate_discriminant")

# Conditional Edge (discriminant check ke baad branch)
builder.add_conditional_edges(
    "calculate_discriminant",
    check_condition,
    {
        "no_real_roots": "no_real_roots",
        "real_roots": "real_roots",
        "repeated_roots": "repeated_roots"
    }
)

# Fan-in to END
builder.add_edge("no_real_roots", END)
builder.add_edge("real_roots", END)
builder.add_edge("repeated_roots", END)

graph = builder.compile()

In [60]:
if __name__ == "__main__":
    initial_input = {
        "a": float(input("Enter a: ")),
        "b": float(input("Enter b: ")),
        "c": float(input("Enter c: "))
    }

    result = graph.invoke(initial_input)
    print("\n--- QUADRATIC EQUATION RESULT ---")
    print(f"Equation: {result['equation']}")
    print(f"Discriminant: {result['discriminant']}")
    print(f"Result: {result['result']}")


--- QUADRATIC EQUATION RESULT ---
Equation: x² + 6x + 9
Discriminant: 0.0
Result: One repeated root: -3.00
